# ROI Preprocessing

Refresh or normalize the canonical processed ROI dataset.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from meatlens_pork_pipeline.image_ops import process_image

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())


In [ ]:
def preprocess_roi_image(path: Path, background_mode: str = 'gray') -> tuple[np.ndarray, dict[str, object]]:
    return process_image(path, background_mode=background_mode)


def build_processed_output_path(row: pd.Series, output_root: Path) -> Path:
    sample_number = str(row.get('sample_number', '')).strip()
    label = str(row['label']).strip()
    image_file_name = str(row['image_file_name']).strip()
    if sample_number:
        return output_root / f'sample {sample_number}' / label / image_file_name
    return output_root / label / image_file_name


def normalize_existing_processed_manifest(audited_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows: list[dict[str, object]] = []
    summary_rows: list[dict[str, object]] = []
    for row in audited_df.to_dict(orient='records'):
        local_path = Path(str(row['local_image_path']))
        record = dict(row)
        record['local_image_path'] = str(local_path)
        record['processed_output_file'] = str(local_path)
        rows.append(record)
        summary_rows.append(
            {
                'image_file_name': row['image_file_name'],
                'sample_number': row.get('sample_number', ''),
                'sample_id': row.get('sample_id', ''),
                'label': row['label'],
                'processed_output_file': str(local_path),
                'segmentation_failed': str(row.get('segmentation_failed', 'False')),
                'mask_area_ratio': row.get('mask_area_ratio', ''),
                'center_overlap_ratio': row.get('center_overlap_ratio', ''),
                'number_of_components': row.get('number_of_components', ''),
                'touches_border': row.get('touches_border', ''),
            }
        )
    return pd.DataFrame(rows), pd.DataFrame(summary_rows), pd.DataFrame(columns=['image_file_name', 'error'])


def preprocess_audited_manifest(
    audited_df: pd.DataFrame,
    output_root: Path,
    background_mode: str = 'gray',
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    records: list[dict[str, object]] = []
    summary_rows: list[dict[str, object]] = []
    failure_rows: list[dict[str, object]] = []

    for row in audited_df.to_dict(orient='records'):
        source_path = Path(str(row['local_image_path']))
        output_path = build_processed_output_path(pd.Series(row), output_root)
        ensure_dir(output_path.parent)
        try:
            output_uint8, metadata = preprocess_roi_image(source_path, background_mode=background_mode)
            Image.fromarray(output_uint8).save(output_path)
            processed_record = dict(row)
            processed_record['local_image_path'] = str(output_path)
            processed_record['processed_output_file'] = str(output_path)
            processed_record.update(metadata)
            records.append(processed_record)
            summary_rows.append(
                {
                    'image_file_name': row['image_file_name'],
                    'sample_number': row.get('sample_number', ''),
                    'sample_id': row.get('sample_id', ''),
                    'label': row['label'],
                    'processed_output_file': str(output_path),
                    **metadata,
                }
            )
            if bool(metadata.get('segmentation_failed', False)):
                failure_rows.append({'image_file_name': row['image_file_name'], 'error': 'segmentation_failed'})
        except Exception as exc:
            failure_rows.append({'image_file_name': row.get('image_file_name', ''), 'error': str(exc)})

    return pd.DataFrame(records), pd.DataFrame(summary_rows), pd.DataFrame(failure_rows)


In [ ]:
AUDITED_MANIFEST_PATH = Path(str(override('AUDITED_MANIFEST_PATH', GENERATED_SPLITS_ROOT / 'audited_manifest.csv')))
PROCESSED_MANIFEST_PATH = Path(str(override('PROCESSED_MANIFEST_PATH', GENERATED_SPLITS_ROOT / 'processed_manifest.csv')))
PREPROCESSING_SUMMARY_PATH = Path(str(override('PREPROCESSING_SUMMARY_PATH', PROCESSED_ROI_ROOT / 'preprocessing_summary.csv')))
PREPROCESSING_FAILURES_PATH = Path(str(override('PREPROCESSING_FAILURES_PATH', PROCESSED_ROI_ROOT / 'preprocessing_failures.csv')))
BACKGROUND_MODE = str(override('BACKGROUND_MODE', 'gray'))
FORCE_REPROCESS = bool(override('FORCE_REPROCESS', False))

audited_df = pd.read_csv(AUDITED_MANIFEST_PATH, dtype=str).fillna('')
ensure_dir(PROCESSED_MANIFEST_PATH.parent)
ensure_dir(PREPROCESSING_SUMMARY_PATH.parent)
ensure_dir(PREPROCESSING_FAILURES_PATH.parent)

use_existing_processed = (
    not FORCE_REPROCESS
    and not audited_df.empty
    and set(audited_df['source_manifest_type'].astype(str).str.strip()) == {'processed_summary'}
)

if use_existing_processed:
    processed_df, summary_df, failures_df = normalize_existing_processed_manifest(audited_df)
else:
    processed_df, summary_df, failures_df = preprocess_audited_manifest(
        audited_df,
        output_root=PROCESSED_ROI_ROOT,
        background_mode=BACKGROUND_MODE,
    )

processed_df.to_csv(PROCESSED_MANIFEST_PATH, index=False)
summary_df.to_csv(PREPROCESSING_SUMMARY_PATH, index=False)
failures_df.to_csv(PREPROCESSING_FAILURES_PATH, index=False)

print(f'Processed rows: {len(processed_df)}')
print(f'Preprocessing failures: {len(failures_df)}')
